# 05 — Train DaViT-CC (CC-View Specialist)

Trains the CC-view specialist using configs/davit_cc.yaml hyperparameters.

**Paper hyperparameters:**
- head: single (LayerNorm → Dropout(0.5) → Linear(1024→2))
- optimizer: AdamW, lr=1e-4, betas=(0.9, 0.9), eps=1e-7, wd=1e-2
- scheduler: ReduceLROnPlateau(factor=0.7, patience=8)
- early stopping: patience=15, composite score 0.5*acc + 0.2*auc + 0.2*f1 + 0.1*(1-loss)

In [ ]:
import torch
from torch.utils.data import DataLoader

from multiview_davit.config import load_config
from multiview_davit.seed import set_seed
from multiview_davit.data.datasets import MedicalImageDataset
from multiview_davit.data.transforms import build_train_transform, build_inference_transform
from multiview_davit.training.multi_run import multi_run_training
from multiview_davit.evaluation.reports import plot_and_save_metrics

set_seed(42)
cfg = load_config('configs/davit_cc.yaml')
data_cfg = load_config('configs/data.yaml')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
train_ds = MedicalImageDataset(data_cfg.cbis_ddsm.train_csv,
                                transform=build_train_transform(cfg.training.input_size))
val_ds   = MedicalImageDataset(data_cfg.cbis_ddsm.val_csv,
                                transform=build_inference_transform(cfg.training.input_size))

train_loader = DataLoader(train_ds, batch_size=cfg.training.batch_size,
                           shuffle=True, num_workers=cfg.training.num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.training.batch_size,
                           shuffle=False, num_workers=cfg.training.num_workers, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
best_model, history, best_score = multi_run_training(
    cfg=cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    checkpoint_path='checkpoints/davit_cc_best.pth',
)
print(f'Best composite score: {best_score:.4f}')
plot_and_save_metrics(history, save_dir='results/', case_name='davit_cc')